# S4.5 — 分类方法准确率对比

基于人工标注 (CMIP6_mean_shape_manual_labels_0920) 评估 4 种分类方法：

| 方法 | Simple 判据 |
|------|-----------|
| M1 | \|Pearson\| ≥ 0.80 |
| M2 | TP=0 且 PL R² ≥ 0.60 (当前方法) |
| M3 | PL R² ≥ 0.85 (所有面板) |
| M4 | \|Pearson\| ≥ 0.80 且 PL R² ≥ 0.85 |

评估范围:  过了 branch 之后、排除 Flat/Unresolved/Insufficient。

In [ ]:
from __future__ import annotations
import sys, warnings, importlib, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl
from IPython.display import display, Image

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

CASE_DIR   = Path('/Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA')
DATA_ROOT  = Path('/Volumes/mimi-T9/CMIP6')
if str(CASE_DIR) not in sys.path:
    sys.path.insert(0, str(CASE_DIR))

EPS_FILTER = 1e-2
EPS_TAG    = '1e-2'
OUTPUT_DIR = CASE_DIR / 'output' / f'S4.4_{EPS_TAG}'
FEAT_PATH  = OUTPUT_DIR / f'classification_{EPS_TAG}.csv'
CURVE_PATH = OUTPUT_DIR / f'lowess_curves_{EPS_TAG}.parquet'
BRANCH_PATH = OUTPUT_DIR / f'branch_results_5bw_{EPS_TAG}.csv'
LABEL_PATH  = CASE_DIR / 'label' / 'CMIP6_mean_shape_manual_labels_0920.xlsx'

ACC_DIR = OUTPUT_DIR / 'accuracy_comparison'
ACC_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ['CESM2', 'CNRM-CM6-1', 'CanESM5', 'GFDL-CM4', 'CMCC-CM2-SR5']
ZONES  = ['all_land', 'WW', 'WD', 'CW', 'CD', 'LI']
VARIABLE_PAIRS = [
    ('P','Q','P → Q'), ('ET','Q','ET → Q'), ('mrros','Q','mrros → Q'),
    ('prsn','Q','prsn → Q'), ('tran','Q','tran → Q'),
    ('evspsblsoi','Q','evspsblsoi → Q'), ('hfls','Q','hfls → Q'),
    ('hfss','Q','hfss → Q'), ('lai','Q','lai → Q'), ('tas','Q','tas → Q'),
    ('rsds','Q','rsds → Q'), ('mrso','Q','mrso → Q'), ('mrsos','Q','mrsos → Q'),
    ('rlds','Q','rlds → Q'), ('rlus','Q','rlus → Q'), ('rsus','Q','rsus → Q'),
    ('P','ET','P → ET'),
]
START_YEAR, END_YEAR = 1985, 2014

VAR_FULL_NAMES = {
    'P':'Precipitation','ET':'Evapotranspiration','Q':'Total Runoff',
    'hfls':'Latent Heat Flux','hfss':'Sensible Heat Flux','tran':'Transpiration',
    'evspsblsoi':'Soil Evaporation','mrros':'Surface Runoff',
    'mrso':'Total Soil Moisture','mrsos':'Topsoil Moisture','lai':'LAI',
    'tas':'Temperature','prsn':'Snowfall','rlds':'Downward LW',
    'rlus':'Upward LW','rsds':'Downward SW','rsus':'Upward SW',
}
def pair_display_title(x_var, y_var):
    xn = VAR_FULL_NAMES.get(x_var, x_var)
    yn = VAR_FULL_NAMES.get(y_var, y_var)
    return f'{xn} ({x_var}) → {yn} ({y_var})'

print(f'Output dir: {OUTPUT_DIR}')
print(f'Label file: {LABEL_PATH}')

In [ ]:
runs = []
for model in MODELS:
    search_roots = [
        DATA_ROOT / model / 'historical',
        CASE_DIR / 'data' / model / 'historical',
    ]
    zone_name = f'zone_climatology_{START_YEAR}_{END_YEAR}.parquet'
    for sr in search_roots:
        if not sr.exists():
            continue
        zone_paths = sorted(sr.glob(f'*/*/land/zones/{zone_name}'))
        if zone_paths:
            t = pd.read_parquet(zone_paths[0])
            t.rename(columns={'R':'Q'}, inplace=True)
            runs.append({'model': model, 'data': t})
            print(f'{model}: {len(t)} rows')
            break
print(f'Loaded {len(runs)} models')

def get_xy(run, x_var, y_var, zone):
    data = run['data']
    if x_var not in data.columns or y_var not in data.columns:
        return None, None
    sub = data if zone == 'all_land' else data[data['analysis_zone'] == zone]
    if len(sub) < 5:
        return None, None
    x = sub[x_var].values.astype(float)
    y = sub[y_var].values.astype(float)
    finite = np.isfinite(x) & np.isfinite(y)
    return x[finite], y[finite]

In [ ]:
import lowess_classifier as lc
importlib.reload(lc)

curves_dict = lc.load_lowess_curves(CURVE_PATH)
print(f'Loaded {len(curves_dict)} curves')

all_features = []
for (pair, model, zone), (xs, ys) in curves_dict.items():
    x_eq = np.linspace(xs.min(), xs.max(), 15)
    y_eq = np.interp(x_eq, xs, ys)
    pearson = float(np.corrcoef(x_eq, y_eq)[0, 1]) if np.ptp(y_eq) > 0 else 0.0
    n_tp_filtered, n_tp_raw = lc._count_turning_points_filtered(y_eq, lc.DEADZONE)
    pl_r2, pl_b = lc._fit_powerlaw_r2(xs, ys)
    B, bow_cls = lc._bow_classify(y_eq)
    endpoint_change = float(y_eq[-1] - y_eq[0])
    direction = 'up' if endpoint_change >= 0 else 'down'
    all_features.append({
        'pair': pair, 'model': model, 'zone': zone,
        'pearson': pearson,
        'n_tp_filtered': n_tp_filtered, 'n_tp_raw': n_tp_raw,
        'pl_r2': pl_r2, 'pl_b': pl_b,
        'bow_B': B, 'bow_cls': bow_cls,
        'direction': direction,
    })

feat_df = pd.DataFrame(all_features)

csv_df = pd.read_csv(FEAT_PATH)
csv_cols = ['Pair','Model','Zone','r2','mic','ls_mas','ls_mev','mic_pass',
            'ls_effect_size','flatness_class','ls_curve_pearson']
feat_df = feat_df.merge(
    csv_df[csv_cols].rename(columns={'Pair':'pair','Model':'model','Zone':'zone'}),
    on=['pair','model','zone'], how='left')

print(f'Feature table: {len(feat_df)} panels')
print(f'PL R2 computed: {feat_df["pl_r2"].notna().sum()} / {len(feat_df)}')

In [ ]:
_sheet_to_pair = {
    'P_Q':'P → Q','ET_Q':'ET → Q','tran_Q':'tran → Q',
    'mrros_Q':'mrros → Q','tas_Q':'tas → Q','prsn_Q':'prsn → Q',
    'evspsblsoi_Q':'evspsblsoi → Q','hfls_Q':'hfls → Q',
    'hfss_Q':'hfss → Q','lai_Q':'lai → Q','rsds_Q':'rsds → Q',
    'mrso_Q':'mrso → Q','mrsos_Q':'mrsos → Q','rlds_Q':'rlds → Q',
    'rlus_Q':'rlus → Q','rsus_Q':'rsus → Q','P_ET':'P → ET',
}

wb = openpyxl.load_workbook(str(LABEL_PATH), read_only=True)
shape_labels, branch_labels, response_labels = {}, {}, {}

for sh_name, pair_name in _sheet_to_pair.items():
    if sh_name not in wb.sheetnames:
        continue
    ws = wb[sh_name]
    for ii, mm in enumerate(MODELS):
        for jj, zz in enumerate(ZONES):
            val = ws.cell(5 + ii, 2 + jj).value
            if val is not None:
                response_labels[(pair_name, mm, zz)] = str(val).strip()
            val = ws.cell(14 + ii, 2 + jj).value
            if val is not None:
                shape_labels[(pair_name, mm, zz)] = val
            val = ws.cell(23 + ii, 2 + jj).value
            if val is not None:
                branch_labels[(pair_name, mm, zz)] = str(val).strip()
wb.close()

def label_to_class(label):
    if label is None:
        return None
    s = str(label).strip()
    if s in ('Monotonic ↑', 'Monotonic ↓'):
        return 'simple'
    if s in ('1','2','3','4','Non-monotonic'):
        return 'complex'
    return None

def label_to_detail(label):
    if label is None:
        return None
    s = str(label).strip()
    if s in ('Monotonic ↑', 'Monotonic ↓'):
        return 'simple'
    if s == '1':
        return 'TP=1'
    if s in ('2','3','4','Non-monotonic'):
        return 'TP≥2'
    return None

feat_df['manual_shape'] = feat_df.apply(
    lambda r: shape_labels.get((r['pair'], r['model'], r['zone'])), axis=1)
feat_df['manual_class'] = feat_df['manual_shape'].map(label_to_class)
feat_df['manual_detail'] = feat_df['manual_shape'].map(label_to_detail)
feat_df['manual_branch'] = feat_df.apply(
    lambda r: branch_labels.get((r['pair'], r['model'], r['zone']), 'No branch'), axis=1)

print(f'Shape labels: {len(shape_labels)}, branch labels: {len(branch_labels)}')
print(f'Manual class distribution:')
print(feat_df['manual_class'].value_counts(dropna=False).to_string())
print()
print(f'Manual detail distribution:')
print(feat_df['manual_detail'].value_counts(dropna=False).to_string())
print()
print(f'Manual branch:')
print(feat_df['manual_branch'].value_counts().to_string())

In [ ]:
import classify_method1 as m1
import classify_method3 as m3
import classify_method4 as m4
importlib.reload(m1); importlib.reload(m3); importlib.reload(m4)

methods = {
    m1.NAME: m1.is_simple,
    'M2: TP=0 + PL R2>=0.60': lambda p, r, t: (t == 0) and np.isfinite(r) and (r >= 0.60),
    m3.NAME: m3.is_simple,
    m4.NAME: m4.is_simple,
}

for name, fn in methods.items():
    feat_df[name] = feat_df.apply(
        lambda row, _fn=fn: 'simple' if _fn(row['pearson'], row['pl_r2'], row['n_tp_filtered']) else 'complex',
        axis=1)

mask = (feat_df['manual_class'].notna()) & (feat_df['manual_branch'] == 'No branch')
eval_df = feat_df[mask].copy()
print(f'Panels for evaluation: {len(eval_df)} / {len(feat_df)}')
print(f'  (excluded: branch, Flat, Unresolved, Insufficient data)\n')

# --- Overall accuracy ---
results = []
for name in methods:
    correct = (eval_df[name] == eval_df['manual_class']).sum()
    total = len(eval_df)
    acc = correct / total * 100
    tp = ((eval_df[name]=='simple') & (eval_df['manual_class']=='simple')).sum()
    fp = ((eval_df[name]=='simple') & (eval_df['manual_class']=='complex')).sum()
    fn = ((eval_df[name]=='complex') & (eval_df['manual_class']=='simple')).sum()
    tn = ((eval_df[name]=='complex') & (eval_df['manual_class']=='complex')).sum()
    prec_s = tp/(tp+fp)*100 if (tp+fp)>0 else 0
    prec_c = tn/(tn+fn)*100 if (tn+fn)>0 else 0
    recall_s = tp/(tp+fn)*100 if (tp+fn)>0 else 0
    recall_c = tn/(tn+fp)*100 if (tn+fp)>0 else 0
    results.append({
        'Method': name,
        'Accuracy': f'{correct}/{total} ({acc:.1f}%)',
        'Simple Prec': f'{prec_s:.1f}%',
        'Simple Recall': f'{recall_s:.1f}%',
        'Complex Prec': f'{prec_c:.1f}%',
        'Complex Recall': f'{recall_c:.1f}%',
    })
    print(f'{name}:')
    print(f'  Accuracy:       {correct}/{total} = {acc:.1f}%')
    print(f'  Simple  — Prec={prec_s:.1f}%  Recall={recall_s:.1f}%  (TP={tp} FP={fp} FN={fn})')
    print(f'  Complex — Prec={prec_c:.1f}%  Recall={recall_c:.1f}%  (TN={tn})\n')

print('=== Overall Accuracy ===')
display(pd.DataFrame(results))

# --- Per-class accuracy ---
print('\n' + '='*70)
print('  Per-class accuracy (分母 = 人工标注的实际个数)')
print('='*70 + '\n')

n_simple = (eval_df['manual_detail'] == 'simple').sum()
n_tp1    = (eval_df['manual_detail'] == 'TP=1').sum()
n_tp2    = (eval_df['manual_detail'] == 'TP≥2').sum()
print(f'Label distribution:  Simple={n_simple},  TP=1={n_tp1},  TP≥2={n_tp2},  Total={n_simple+n_tp1+n_tp2}\n')

per_class_rows = []
for name in methods:
    mask_simple = eval_df['manual_detail'] == 'simple'
    mask_tp1    = eval_df['manual_detail'] == 'TP=1'
    mask_tp2    = eval_df['manual_detail'] == 'TP≥2'

    correct_simple = (eval_df.loc[mask_simple, name] == 'simple').sum()
    correct_tp1    = (eval_df.loc[mask_tp1,    name] == 'complex').sum()
    correct_tp2    = (eval_df.loc[mask_tp2,    name] == 'complex').sum()

    acc_simple = correct_simple / n_simple * 100 if n_simple > 0 else 0
    acc_tp1    = correct_tp1 / n_tp1 * 100 if n_tp1 > 0 else 0
    acc_tp2    = correct_tp2 / n_tp2 * 100 if n_tp2 > 0 else 0

    per_class_rows.append({
        'Method': name,
        f'Simple ({n_simple})': f'{correct_simple}/{n_simple} ({acc_simple:.1f}%)',
        f'TP=1 ({n_tp1})': f'{correct_tp1}/{n_tp1} ({acc_tp1:.1f}%)',
        f'TP≥2 ({n_tp2})': f'{correct_tp2}/{n_tp2} ({acc_tp2:.1f}%)',
    })

    print(f'{name}:')
    print(f'  Simple:  {correct_simple}/{n_simple} = {acc_simple:.1f}%')
    print(f'  TP=1:    {correct_tp1}/{n_tp1} = {acc_tp1:.1f}%')
    print(f'  TP≥2:    {correct_tp2}/{n_tp2} = {acc_tp2:.1f}%')
    print()

print('=== Per-class Accuracy ===')
display(pd.DataFrame(per_class_rows))

In [ ]:
pair_acc = []
for pair in [p[2] for p in VARIABLE_PAIRS]:
    sub = eval_df[eval_df['pair'] == pair]
    if len(sub) == 0:
        continue
    row = {'Pair': pair, 'N': len(sub)}
    for name in methods:
        correct = (sub[name] == sub['manual_class']).sum()
        row[name] = f'{correct}/{len(sub)} ({correct/len(sub)*100:.0f}%)'
    pair_acc.append(row)

pair_df = pd.DataFrame(pair_acc)
display(pair_df)

In [ ]:
for name in methods:
    errors = eval_df[eval_df[name] != eval_df['manual_class']].copy()
    if len(errors) == 0:
        print(f'{name}: no errors')
        continue
    print(f'\n=== {name}: {len(errors)} errors ===')
    for _, r in errors.iterrows():
        pred = r[name]
        true = r['manual_class']
        ms = r['manual_shape']
        print(f'  {r["pair"]:25s} {r["model"]:20s} {r["zone"]:10s}  '
              f'pred={pred:7s} true={true:7s} (label={ms})  '
              f'rho={r["pearson"]:+.2f} TP={int(r["n_tp_filtered"])} R2={r["pl_r2"]:.3f}')

In [ ]:
import matplotlib
if str(matplotlib.get_backend()).lower() == 'agg':
    try:
        matplotlib.use('module://matplotlib_inline.backend_inline', force=True)
    except Exception:
        pass
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, to_rgba
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

LOWESS_COLOR = '#B71C1C'
SHAPE_BASE_COLORS = {
    'linear':'#d4960a','saturation':'#eb6834','acceleration':'#e87ba4',
    'sigmoid':'#c2185b','cubic':'#7b1fa2',
    'TP=0':'#2a78d6','TP=1':'#0097a7','TP>=2':'#008300',
}
BRANCH_COLOR = '#6250d6'
GRAY_COLOR   = '#898781'
UNIT_LABELS = {
    'P':'Precipitation (mm/yr)','ET':'Evapotranspiration (mm/yr)',
    'Q':'Total runoff (mm/yr)','hfls':'Latent heat flux (W/m2)',
    'hfss':'Sensible heat flux (W/m2)','tran':'Transpiration (mm/yr)',
    'evspsblsoi':'Soil evaporation (mm/yr)','mrros':'Surface runoff (mm/yr)',
    'mrso':'Total soil moisture (kg/m2)','mrsos':'Topsoil moisture (kg/m2)',
    'lai':'LAI (m2/m2)','tas':'Temperature (K)',
    'prsn':'Snowfall (mm/yr)','rlds':'Downward LW (W/m2)',
    'rlus':'Upward LW (W/m2)','rsds':'Downward SW (W/m2)',
    'rsus':'Upward SW (W/m2)',
}
ZONE_SHORT = {
    'all_land':'All land','WW':'Wet-warm (WW)','WD':'Dry-warm (WD)',
    'CW':'Wet-cold (CW)','CD':'Dry-cold (CD)','LI':'Land ice (LI)',
}

if BRANCH_PATH.exists():
    _br_df = pd.read_csv(BRANCH_PATH)
    branch_lookup = {(r['Pair'],r['Model'],r['Zone']): r['group'] for _,r in _br_df.iterrows()}
else:
    branch_lookup = {}

R2_ALPHA_REF = 0.9

def _r2_alpha(r2, alpha_min=0.15):
    if r2 is None or not np.isfinite(r2):
        return alpha_min
    return float(np.clip(r2 / R2_ALPHA_REF, alpha_min, 1.0))

def build_method_lookup(method_name, feat_df_in):
    lookup = {}
    for _, row in feat_df_in.iterrows():
        key = (row['pair'], row['model'], row['zone'])
        is_simple = (row[method_name] == 'simple')
        mic_pass = bool(row.get('mic_pass', True))
        r2 = row.get('r2', np.nan)
        bg = branch_lookup.get(key, 'No branch')
        if not mic_pass:
            fc = 'low_mic'
        elif bg in ('Branch', 'Candidate branch'):
            fc = bg
        elif is_simple:
            fc = 'simple'
        else:
            fc = 'complex'
        if is_simple:
            shape2 = row['bow_cls']
            da = '\u2191' if row['direction'] == 'up' else '\u2193'
            sl = f"{shape2}{da}"
        else:
            tp = int(row['n_tp_filtered'])
            if tp == 0:
                shape2 = 'TP=0'
                da = '\u2191' if row['direction'] == 'up' else '\u2193'
                sl = f'TP=0{da}'
            elif tp == 1:
                shape2 = 'TP=1'; sl = 'TP=1'
            else:
                shape2 = 'TP>=2'; sl = 'TP>=2'
        lookup[key] = {
            'lowess_shape': 'simple' if is_simple else 'complex',
            'lowess_shape2': shape2, 'shape_label': sl,
            'r2': r2, 'mic_pass': mic_pass, 'final_class': fc,
            'branch_group': bg, 'mic': row.get('mic',np.nan),
            'ls_mas': row.get('ls_mas',np.nan), 'ls_mev': row.get('ls_mev',np.nan),
            'ls_pearson': row['pearson'], 'ls_powerlaw_r2': row['pl_r2'],
        }
    return lookup

def column_limits(x_var, y_var, zone):
    xs_all, ys_all = [], []
    for run in runs:
        x, y = get_xy(run, x_var, y_var, zone)
        if x is None: continue
        m = (np.abs(x)>EPS_FILTER)&(np.abs(y)>EPS_FILTER)
        if m.sum()<5: continue
        xs_all.append(x[m]); ys_all.append(y[m])
    if not xs_all: return (0,1),(0,1)
    xa=np.concatenate(xs_all); ya=np.concatenate(ys_all)
    def lim(a):
        lo,hi=float(np.min(a)),float(np.max(a))
        mg=(hi-lo)*0.05
        return (lo-mg,hi+mg)
    return lim(xa),lim(ya)

def _subtitle(row):
    if row is None: return ['no data'],[GRAY_COLOR]
    mic=row.get('mic',np.nan); r2=row.get('r2',np.nan)
    pl_r2=row.get('ls_powerlaw_r2',np.nan); pear=row.get('ls_pearson',np.nan)
    mic_str=f'MIC={mic:.2f}' if np.isfinite(mic) else ''
    r2_str=f'R2={r2:.2f}' if np.isfinite(r2) else ''
    if np.isfinite(pl_r2): r2_str+=f' PL={pl_r2:.2f}'
    corr_str=f'rho={pear:+.2f}' if np.isfinite(pear) else ''
    if not row.get('mic_pass',True): return [mic_str or 'MIC=na'],[GRAY_COLOR]
    bg=row.get('branch_group','')
    if bg in ('Branch','Candidate branch'):
        lines=[l for l in [mic_str,r2_str,corr_str] if l]
        return lines,[BRANCH_COLOR]*len(lines)
    s2=row.get('lowess_shape2','')
    color=SHAPE_BASE_COLORS.get(s2,GRAY_COLOR)
    lines=[l for l in [mic_str,r2_str,corr_str] if l]
    return lines,[color]*len(lines)

def _panel_facecolor(row):
    if row is None: return to_rgba(GRAY_COLOR,0.08)
    fc=row.get('final_class','')
    if fc=='low_mic': return to_rgba(GRAY_COLOR,0.08)
    if fc in ('Branch','Candidate branch'): return to_rgba(BRANCH_COLOR,0.20)
    s2=row.get('lowess_shape2',''); r2=row.get('r2',np.nan)
    if s2 in SHAPE_BASE_COLORS: return to_rgba(SHAPE_BASE_COLORS[s2],_r2_alpha(r2)*0.45)
    return to_rgba(GRAY_COLOR,0.08)

def _panel_border(ax, row):
    if row is None: c=to_rgba(GRAY_COLOR,0.3)
    else:
        fc=row.get('final_class',''); s2=row.get('lowess_shape2',''); r2=row.get('r2',np.nan)
        if fc=='low_mic': c=to_rgba(GRAY_COLOR,0.3)
        elif fc in ('Branch','Candidate branch'): c=to_rgba(BRANCH_COLOR,0.7)
        elif s2 in SHAPE_BASE_COLORS: c=to_rgba(SHAPE_BASE_COLORS[s2],_r2_alpha(r2))
        else: c=to_rgba(GRAY_COLOR,0.3)
    for spine in ax.spines.values(): spine.set_color(c); spine.set_linewidth(2.0)

def _shape_label_text(row):
    if row is None: return '',GRAY_COLOR
    bg=row.get('branch_group','')
    if bg in ('Branch','Candidate branch'):
        return ('branch' if bg=='Branch' else 'cand. branch'),BRANCH_COLOR
    if not row.get('mic_pass',True): return '',GRAY_COLOR
    sl=row.get('shape_label',''); s2=row.get('lowess_shape2','')
    return sl,SHAPE_BASE_COLORS.get(s2,GRAY_COLOR)

print('Plotting helpers loaded.')

In [ ]:
COL_W, ROW_H, SUB_H = 3.2, 3.0, 1.0

def make_hexbin_figure(x_var, y_var, pair_label, save_dir, lookup, method_title):
    n_rows, n_cols = len(MODELS), len(ZONES)
    col_lims = {z: column_limits(x_var,y_var,z) for z in ZONES}
    fig = plt.figure(figsize=((COL_W+0.55)*n_cols+1.4,(SUB_H+ROW_H+0.30)*n_rows+1.0))
    outer = GridSpec(n_rows,n_cols,figure=fig,hspace=0.10,wspace=0.25,
                     left=0.08,right=0.99,top=0.92,bottom=0.06)
    dt = pair_display_title(x_var, y_var)
    fig.suptitle(f'{method_title}\nhexbin  {dt}', fontsize=13, fontweight='semibold')
    for ri,run in enumerate(runs):
        model = run['model']
        for cj,zone in enumerate(ZONES):
            xlim,ylim = col_lims[zone]
            inner = outer[ri,cj].subgridspec(2,1,height_ratios=[SUB_H,ROW_H],hspace=0.02)
            ax_sub=fig.add_subplot(inner[0,0]); ax=fig.add_subplot(inner[1,0])
            ax_sub.axis('off')
            if ri==0: ax_sub.set_title(ZONE_SHORT[zone],fontsize=13,fontweight='semibold',pad=4)
            if cj==0:
                ax.set_ylabel(UNIT_LABELS.get(y_var,y_var),fontsize=10)
                ax.text(-0.35,0.5,model,transform=ax.transAxes,fontsize=12,
                        fontweight='bold',ha='right',va='center',rotation=90)
            if ri==n_rows-1: ax.set_xlabel(UNIT_LABELS.get(x_var,x_var),fontsize=10)
            x,y = get_xy(run,x_var,y_var,zone)
            row = lookup.get((pair_label,model,zone))
            if x is None or len(x)<5:
                ax.text(0.5,0.5,'no data',ha='center',va='center',
                        transform=ax.transAxes,fontsize=10,color='#999'); continue
            m=(np.abs(x)>EPS_FILTER)&(np.abs(y)>EPS_FILTER); xc,yc=x[m],y[m]
            if len(xc)<40:
                ax.text(0.5,0.5,f'Too few (N={len(xc)})',ha='center',va='center',
                        transform=ax.transAxes,fontsize=10,color='#C03030'); continue
            ax.hexbin(xc,yc,gridsize=26,cmap='Greys',mincnt=1,norm=LogNorm(),
                      extent=[xlim[0],xlim[1],ylim[0],ylim[1]],linewidths=0.1,zorder=2)
            key=(pair_label,model,zone)
            if key in curves_dict:
                xs_c,ys_c=curves_dict[key]
                ax.plot(xs_c,ys_c,color='white',lw=3.2,zorder=6)
                ax.plot(xs_c,ys_c,color=LOWESS_COLOR,lw=1.0,zorder=7)
            sl_text,sl_color=_shape_label_text(row)
            if sl_text:
                ax.text(0.04,0.96,sl_text,transform=ax.transAxes,ha='left',va='top',
                        fontsize=9,fontweight='bold',color=sl_color,
                        bbox=dict(boxstyle='round,pad=0.15',fc='white',ec=sl_color,alpha=0.85),zorder=10)
            sub_lines,sub_colors=_subtitle(row)
            nl=len(sub_lines)
            yp=[0.78,0.50,0.22][:nl] if nl>=3 else ([0.65,0.25] if nl==2 else [0.50])
            for txt,clr,ypos in zip(sub_lines,sub_colors,yp):
                if txt: ax_sub.text(0.5,ypos,txt,transform=ax_sub.transAxes,ha='center',
                                     va='center',fontsize=9,fontfamily='monospace',color=clr)
            ax.set_facecolor(_panel_facecolor(row)); _panel_border(ax,row)
            ax.set_xlim(xlim); ax.set_ylim(ylim); ax.tick_params(labelsize=9)
    handles=[Line2D([0],[0],color=LOWESS_COLOR,lw=1.8,label='LOWESS')]
    for lab,k in [('linear','linear'),('saturation','saturation'),('acceleration','acceleration'),
                   ('sigmoid','sigmoid'),('cubic','cubic'),('TP=0','TP=0'),('TP=1','TP=1'),('TP>=2','TP>=2')]:
        c=SHAPE_BASE_COLORS[k]
        handles.append(mpatches.Patch(facecolor=to_rgba(c,0.45),edgecolor=c,linewidth=1.2,label=lab))
    handles.append(mpatches.Patch(facecolor=to_rgba(BRANCH_COLOR,0.20),edgecolor=BRANCH_COLOR,linewidth=1.2,label='branch'))
    handles.append(mpatches.Patch(facecolor=to_rgba(GRAY_COLOR,0.08),edgecolor=to_rgba(GRAY_COLOR,0.3),linewidth=1.2,label='MIC<0.2'))
    fig.legend(handles=handles,loc='center right',ncol=1,fontsize=11,frameon=True,
               framealpha=0.7,edgecolor='#CCCCCC',bbox_to_anchor=(1.10,0.5))
    pair_slug=f'{x_var}_{y_var}'
    save_path=save_dir/f'hexbin_{pair_slug}.png'
    fig.savefig(save_path,dpi=160,bbox_inches='tight'); plt.close(fig)
    return save_path

CURVE_COL_W, CURVE_ROW_H, CURVE_SUB_H = 2.8, 2.2, 0.9

def make_lowess_figure(x_var, y_var, pair_label, save_dir, lookup, method_title):
    n_rows, n_cols = len(MODELS), len(ZONES)
    fig = plt.figure(figsize=((CURVE_COL_W+0.45)*n_cols+1.2,
                               (CURVE_SUB_H+CURVE_ROW_H+0.25)*n_rows+1.0))
    outer = GridSpec(n_rows,n_cols,figure=fig,hspace=0.12,wspace=0.30,
                     left=0.07,right=0.99,top=0.92,bottom=0.06)
    dt = pair_display_title(x_var, y_var)
    fig.suptitle(f'{method_title}\nLOWESS curves  {dt}', fontsize=12, fontweight='semibold')
    for ri,run in enumerate(runs):
        model = run['model']
        for cj,zone in enumerate(ZONES):
            inner=outer[ri,cj].subgridspec(2,1,height_ratios=[CURVE_SUB_H,CURVE_ROW_H],hspace=0.02)
            ax_sub=fig.add_subplot(inner[0,0]); ax=fig.add_subplot(inner[1,0])
            ax_sub.axis('off')
            if ri==0: ax_sub.set_title(ZONE_SHORT[zone],fontsize=12,fontweight='semibold',pad=3)
            if cj==0:
                ax.text(-0.30,0.5,model,transform=ax.transAxes,fontsize=11,
                        fontweight='bold',ha='right',va='center',rotation=90)
            key=(pair_label,model,zone); row=lookup.get(key)
            if key not in curves_dict:
                ax.text(0.5,0.5,'no curve',ha='center',va='center',
                        transform=ax.transAxes,fontsize=9,color='#999')
                ax.set_xticks([]); ax.set_yticks([]); continue
            xs_c,ys_c=curves_dict[key]
            s2=row.get('lowess_shape2','') if row else ''
            base_color=SHAPE_BASE_COLORS.get(s2,LOWESS_COLOR)
            fc=row.get('final_class','') if row else ''
            if fc=='low_mic': base_color=GRAY_COLOR
            elif fc in ('Branch','Candidate branch'): base_color=BRANCH_COLOR
            ax.plot(xs_c,ys_c,color=base_color,lw=2.2,zorder=5)
            tp_pos=lc.get_tp_positions(xs_c,ys_c)
            if tp_pos['kept']:
                tx,ty=zip(*tp_pos['kept'])
                ax.scatter(tx,ty,marker='o',s=30,c=base_color,edgecolors='white',linewidths=1.0,zorder=8)
            if tp_pos['removed']:
                tx,ty=zip(*tp_pos['removed'])
                ax.scatter(tx,ty,marker='x',s=25,c='#999999',linewidths=1.0,zorder=8)
            ax.axhline(ys_c.mean(),color='#999',ls=':',lw=0.6,alpha=0.5)
            yr=ys_c.max()-ys_c.min()
            if yr>0: ax.set_ylim(ys_c.min()-yr*0.15,ys_c.max()+yr*0.15)
            ax.tick_params(labelsize=7)
            ax.set_facecolor(_panel_facecolor(row)); _panel_border(ax,row)
            sl_text,sl_color=_shape_label_text(row)
            if sl_text:
                ax.text(0.04,0.96,sl_text,transform=ax.transAxes,ha='left',va='top',
                        fontsize=8,fontweight='bold',color=sl_color,
                        bbox=dict(boxstyle='round,pad=0.15',fc='white',ec=sl_color,alpha=0.85),zorder=10)
            sub_lines,sub_colors=_subtitle(row)
            nl=len(sub_lines)
            yp=[0.78,0.50,0.22][:nl] if nl>=3 else ([0.65,0.25] if nl==2 else [0.50])
            for txt,clr,ypos in zip(sub_lines,sub_colors,yp):
                if txt: ax_sub.text(0.5,ypos,txt,transform=ax_sub.transAxes,ha='center',
                                     va='center',fontsize=8,fontfamily='monospace',color=clr)
    pair_slug=f'{x_var}_{y_var}'
    save_path=save_dir/f'lowess_{pair_slug}.png'
    fig.savefig(save_path,dpi=140,bbox_inches='tight'); plt.close(fig)
    return save_path

print('Plot functions loaded.')

In [ ]:
method_short = {
    m1.NAME: m1.SHORT,
    'M2: TP=0 + PL R2>=0.60': 'M2_current',
    m3.NAME: m3.SHORT,
    m4.NAME: m4.SHORT,
}

for method_name in methods:
    short = method_short[method_name]
    method_dir = ACC_DIR / short
    method_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n{"="*60}')
    print(f'  {method_name}')
    print(f'{"="*60}')
    lookup = build_method_lookup(method_name, feat_df)
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        print(f'  {pair_label} ... ', end='', flush=True)
        hp = make_hexbin_figure(x_var,y_var,pair_label,method_dir,lookup,method_name)
        lp = make_lowess_figure(x_var,y_var,pair_label,method_dir,lookup,method_name)
        print(f'done')
        display(Image(filename=str(hp), width=900))
        display(Image(filename=str(lp), width=900))

print(f'\nAll plots saved to {ACC_DIR}')